# Stage 2.3 — MLP (Bengio 2003)

Bigram was hitting a ceiling because the model only sees one character of context. Bengio's 2003 [A Neural Probabilistic Language Model](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) fixed this with two key moves:

1. **Embed each context character into a learned vector** (dimension `d`, e.g. 2 or 10)
2. **Concatenate the embeddings of the last `block_size` characters** and feed to an MLP

Now context is `block_size` characters, embeddings live in continuous space (so similar characters can have similar vectors), and a hidden layer can learn non-linear interactions.

**Target NLL**: ≈ **2.10** (down from 2.46). This is where the line on the loss-vs-architecture chart bends down meaningfully.

## What you'll ship

1. **train/dev/test split** (80/10/10) of the names, on the bigram-pair level
2. **Block-size context windows**: instead of `(prev, next)`, build `(prev_3, prev_2, prev_1) → next`
3. **Embedding table `C`** of shape `(V, embed_dim)`
4. **MLP**: `concat → hidden (tanh) → logits → cross_entropy`
5. **Mini-batch training**: stochastic, batch=32, ~200k iters
6. **Loss-curve plot**: train vs dev, watch for the overfitting inflection
7. **Samples**

## Reference

[Karpathy — Building makemore Part 2: MLP](https://www.youtube.com/watch?v=TCH_1BHY58I)

## Common traps

1. **Block size = 3 means the input is `embed_dim * 3` numbers, not 3.** Easy to size the first hidden layer wrong (it takes `embed_dim * block_size` as input, not `block_size`).
2. **`emb.view(N, -1)`** is how you flatten the `(N, block_size, embed_dim)` tensor into `(N, block_size * embed_dim)`. Use `.view`, not `.reshape` initially — Karpathy uses `view` everywhere.
3. **Mini-batches matter here.** Training on the full ~228k examples per step is slow. Karpathy uses batch=32 randomly-sampled indices per iter.
4. **The lr schedule**: start at 0.1 for ~100k steps, drop to 0.01 for the next 100k. Karpathy explicitly tunes this in the lecture by plotting loss vs LR.
5. **Dev set is for picking the model, NOT touching the test set yet.** Test set goes in stage 2.5.

## Plan

Cells you need to write, in order:

1. **Imports + vocab** — same as 2.1/2.2
2. **Build dataset with `block_size=3`**:
   ```python
   def build_dataset(words, block_size=3):
       X, Y = [], []
       for w in words:
           context = [0] * block_size  # padding
           for ch in w + '.':
               ix = stoi[ch]
               X.append(context); Y.append(ix)
               context = context[1:] + [ix]   # slide window
       return torch.tensor(X), torch.tensor(Y)
   ```
   Then 80/10/10 split via `random.shuffle(words)` first.
3. **Init parameters**:
   - `C` of shape `(V, embed_dim)` — start with `embed_dim=10`
   - `W1` of shape `(embed_dim * block_size, hidden)` — start with `hidden=200`
   - `b1` of shape `(hidden,)`
   - `W2` of shape `(hidden, V)`
   - `b2` of shape `(V,)`
   - All `requires_grad=True`
4. **Forward**: `emb = C[Xb]` → `h = (emb.view(N, -1) @ W1 + b1).tanh()` → `logits = h @ W2 + b2` → `loss = F.cross_entropy(logits, Yb)`
5. **Train loop**: ~200k iters, batch=32, two-stage LR (0.1 then 0.01)
6. **Plot loss curve** (training loss every 1k iters)
7. **Compute final train + dev loss** on the full sets — should both be ~2.10
8. **Sample 10 names** with the trained model

In [ ]:
import random
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

# Vocab — same as previous stages
words = Path('data/names.txt').read_text().splitlines()
chars = sorted(set(''.join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)

# TODO: implement build_dataset, init params, train, sample.

## End-of-stage check

- [ ] Train and dev loss both ≈ 2.10 (gap < 0.05 — no significant overfitting)
- [ ] Loss curve shows clean decrease, no spikes
- [ ] 10 samples look noticeably more name-shaped than Stage 2.1's
- [ ] You can read your `emb.view(N, -1)` call and explain what shape goes in, what shape comes out, and why

One line to `NOTES.md`: what jumped 2.10's loss above 2.46's? (Hint: it's not just having a hidden layer.)

Next: Stage 2.4 — manual backprop on a 6-layer net. Brutal, brilliant.